In [ ]:
#poetry add mhcflurry 'tensorflow==2.18.0' 'ml-dtypes==0.4.1' 'jax==0.5.2'

In [ ]:
import subprocess
import pandas as pd
from pathlib import Path
from mhcflurry import Class1PresentationPredictor

def load_optitype_results(results_dir):
    results_dir = Path(results_dir)
    optitype_files = results_dir.glob('*/*.tsv')

    df_list = []
    for f in optitype_files:
        df = pd.read_csv(f, sep='\t',index_col=0)
        filename = f.stem.replace('_result','')  # e.g., "data.tsv" -> "data"
        df.index = [filename] * len(df)
        df_list.append(df)

    df_alleles = pd.concat(df_list)
    return df_alleles

### Params

In [ ]:
MODALITY = 'WES'
RESULTS_DIR = Path(f'/home/sagemaker-user/custom-file-systems/efs/fs-09913c1f7db79b6fd/results/{MODALITY}/hlatyping/results/')
OUT_FILE = RESULTS_DIR / f'MHCflurry/MHCflurry_predictions.parquet'

### Inputs

In [ ]:
df_alleles = load_optitype_results(RESULTS_DIR / 'optitype')
df_peptides = pd.read_csv('../../gbmcollaboration/metadata/epitope_table_export_1752060444.csv',skiprows=1)

# reformat for MHCFlurry
max_peptide_len = 15
alleles = {idx: row.tolist() for idx, row in df_alleles[['A1','A2','B1','B2','C1','C2']].iterrows()}
monoalleles = {
    patient+' | '+ allele: [allele]
    for patient, allele_list in alleles.items()
    for allele in allele_list
} # split per patient allele list into unique entry to get prediction for each allele, not just best allele 

peptides = df_peptides['Name'].tolist()
peptides_filtered = [p for p in peptides if len(p)<=max_peptide_len]

print(f'Keeping {len(peptides_filtered)} peptides with length<={max_peptide_len} out of {len(peptides)}')

### MHCFlurry

In [ ]:
predictor = Class1PresentationPredictor.load()

# predict best allele only
# df_mhcflurry = predictor.predict(peptides=peptides_filtered,alleles=alleles)

# predict for each allele
df_mhcflurry = predictor.predict(
    peptides=peptides_filtered,
    alleles=monoalleles,
    verbose=0
)

df_mhcflurry[['patient_id','allele']] = pd.DataFrame(
    df_mhcflurry['sample_name'].str.split(' | ').tolist(), index=df_mhcflurry.index
)[[0,2]]

# save
OUT_FILE.parent.mkdir(parents=True,exist_ok=True)
df_mhcflurry.to_parquet(OUT_FILE)